<a href="https://colab.research.google.com/github/roygod1997-wq/code/blob/main/icon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Debugging Matching Logic

Here is the code for the `RUN_03_MATCH_ALL_PRODUCTS` function and its immediate helper functions that handle the matching logic. Please review this code to debug the issue you're encountering.

In [1]:
// Reconstructed from the deleted cell - for review only.

function RUN_03_MATCH_ALL_PRODUCTS() {
  var ctx = getContext_();
  var scaffoldScan = readJson_(ctx.controlFolder, CT.SCAFFOLD_SCAN_FILE);
  var archiveScan = readJson_(ctx.controlFolder, CT.ARCHIVE_SCAN_FILE);

  if (!scaffoldScan) throw new Error('Missing scaffold scan. Run RUN_01_SCAN_FULL_SCAFFOLD.');
  if (!archiveScan) throw new Error('Missing archive scan. Run RUN_02_SCAN_FULL_ARCHIVE.');

  var matches = {
    matched_at: new Date().toISOString(),
    item_count: scaffoldScan.items.length,
    items: []
  };

  for (var i = 0; i < scaffoldScan.items.length; i++) {
    var item = scaffoldScan.items[i];
    var scored = scoreScannedSourcesForItem_(item, archiveScan.files);
    var proof = pickBestProofFromData_(scored);
    var finalCandidates = pickFinalCandidatesFromData_(scored);

    var matchStatus = CT.STATUS.SOURCE_FOUND;
    if (!proof && finalCandidates.length === 0) matchStatus = CT.STATUS.MISSING;
    else if (scored.length > 8) matchStatus = CT.STATUS.DUPLICATE;

    matches.items.push({
      item_code: item.item_code,
      item_name: item.item_name,
      section_name: item.section_name,
      item_folder_id: item.item_folder_id,
      item_folder_url: item.item_folder_url,
      item_card_url: item.item_card_url,
      match_status: matchStatus,
      best_proof: proof,
      final_candidates: finalCandidates,
      top_candidates: scored.slice(0, 15)
    });
  }

  writeJson_(ctx.controlFolder, CT.MATCHES_FILE, matches);
  createText_(ctx.controlFolder, '03_PRODUCT_MATCHES__' + stamp_() + '.md', matchesMarkdown_(matches));

  return {
    status: 'MATCHING_COMPLETE',
    itemCount: matches.items.length
  };
}

function scoreScannedSourcesForItem_(item, files) {
  var scored = [];
  var itemName = normalize_(item.item_name);
  var tokens = itemName.split(' ');
  var itemCode = item.item_code;

  for (var i = 0; i < files.length; i++) {
    var src = files[i];
    var raw = String(src.source_path + ' ' + src.file_name).toLowerCase();
    var full = normalize_(src.source_path + ' ' + src.file_name);
    var ext = getExtension_(src.file_name);

    var score = 0;
    var reasons = [];

    if (hasCodeMatch_(raw, itemCode)) {
      score += 200;
      reasons.push('item code match');
    }

    if (itemName.length > 6 && full.indexOf(itemName) !== -1) {
      score += 160;
      reasons.push('full item name match');
    }

    var hits = 0;
    for (var t = 0; t < tokens.length; t++) {
      var token = tokens[t];
      if (token.length < 4) continue;
      if (isGenericToken_(token)) continue;
      if (full.indexOf(token) !== -1) hits++;
    }

    if (hits > 0) {
      score += hits * 20;
      reasons.push(String(hits) + ' item token hits');
    }

    if (ext === 'png' || ext === 'jpg' || ext === 'jpeg') {
      score += 12;
      reasons.push('visual proof candidate');
    }

    if (ext === 'pdf') {
      score += 10;
      reasons.push('print-ready candidate');
    }

    // Favor exact production clues.
    if (full.indexOf('300dpi') !== -1 || full.indexOf('300 dpi') !== -1) {
      score += 20;
      reasons.push('300 DPI clue');
    }

    if (full.indexOf('us letter') !== -1 || full.indexOf('letter') !== -1) {
      score += 8;
      reasons.push('US Letter clue');
    }

    if (score > 0) {
      scored.push({
        file_id: src.file_id,
        file_name: src.file_name,
        file_url: src.file_url,
        source_path: src.source_path,
        source_bucket: src.source_bucket,
        extension: ext,
        score: score,
        reasons: reasons
      });
    }
  }

  scored.sort(function(a, b) {
    return b.score - a.score;
  });

  return scored;
}


function pickBestProofFromData_(candidates) {
  for (var i = 0; i < candidates.length; i++) {
    if (CT.PROOF_EXTENSIONS.indexOf(candidates[i].extension) !== -1) return candidates[i];
  }
  return null;
}


function pickFinalCandidatesFromData_(candidates) {
  var out = [];
  for (var i = 0; i < candidates.length; i++) {
    if (CT.FINAL_EXTENSIONS.indexOf(candidates[i].extension) !== -1) out.push(candidates[i]);
    if (out.length >= 12) break;
  }
  return out;
}


function firstByExt_(candidates, ext) {
  if (!candidates) return null;
  for (var i = 0; i < candidates.length; i++) {
    if (getExtension_(candidates[i].file_name) === ext) return candidates[i];
  }
  return null;
}


function firstImage_(candidates) {
  if (!candidates) return null;
  for (var i = 0; i < candidates.length; i++) {
    var e = getExtension_(candidates[i].file_name);
    if (e === 'png' || e === 'jpg' || e === 'jpeg') return candidates[i];
  }
  return null;
}


function firstByText_(candidates, terms) {
  if (!candidates) return null;
  for (var i = 0; i < candidates.length; i++) {
    var n = normalize_(candidates[i].file_name + ' ' + candidates[i].source_path);
    for (var t = 0; t < terms.length; t++) {
      if (n.indexOf(normalize_(terms[t])) !== -1) return candidates[i];
    }
  }
  return null;
}


function RUN_05_BUILD_ALL_APPROVED() {
  var ctx = getContext_();
  var matches = readJson_(ctx.controlFolder, CT.MATCHES_FILE);
  if (!matches) throw new Error('Missing matches file. Run RUN_03_MATCH_ALL_PRODUCTS.');

  var sheet = getApprovalSheet_(ctx.controlFolder);
  if (!sheet) throw new Error('Approval sheet not found. Run RUN_04_CREATE_ALL_VISUAL_PROOFS_AND_APPROVAL_QUEUE.');

  var approvals = readApprovalRows_(sheet);
  var approvalMap = {};
  for (var a = 0; a < approvals.length; a++) {
    approvalMap[approvals[a].item_code] = approvals[a];
  }

  var approvedStatuses = [CT.APPROVAL_STATUS.APPROVED, CT.APPROVAL_STATUS.GOOD, CT.APPROVAL_STATUS.BUILT_READY];
  var approvedCount = 0;
  var itemsToBuild = []; // New array to store items that are actually approved and will be built

  for (var i = 0; i < matches.items.length; i++) {
    var item = matches.items[i];
    var approval = approvalMap[item.item_code];

    if (approval && approvedStatuses.indexOf(String(approval.approval_status).toUpperCase()) !== -1) { // Convert to uppercase for robust comparison
      approvedCount++;
      itemsToBuild.push(item);
    }
  }

  var build = {
    built_at: new Date().toISOString(),
    approved_items_found: approvedCount,
    results: []
  };

  if (approvedCount === 0) {
    var noBuildMessage = 'No approved items found in the approval queue. Skipping build and verification steps.';
    createText_(ctx.controlFolder, '05_BUILD_ALL_APPROVED__' + stamp_() + '.md', noBuildReportMarkdown_({ message: noBuildMessage, approved_items_found: 0 }));
    return {
      status: 'NO_APPROVED_ITEMS_TO_BUILD',
      builtCount: 0,
      approvedItemsFound: 0
    };
  }

  for (var i = 0; i < itemsToBuild.length; i++) {
    var item = itemsToBuild[i];
    var folder = DriveApp.getFolderById(item.item_folder_id);
    var folders = getScaffoldItemFolders_(folder);
    var result = buildApprovedItem_(item, folder, folders);
    build.results.push(result);
  }

  writeJson_(ctx.controlFolder, CT.LAST_BUILD_FILE, build);
  createText_(ctx.controlFolder, '05_BUILD_ALL_APPROVED__' + stamp_() + '.md', buildReportMarkdown_(build));

  updateApprovalSheetAfterBuild_(sheet, build.results);

  return {
    status: 'BUILD_APPROVED_COMPLETE',
    builtCount: build.results.length,
    approvedItemsFound: approvedCount
  };
}

function buildReportMarkdown_(build) {
  var lines = [];
  lines.push('# BUILD ALL APPROVED REPORT');
  lines.push('');
  lines.push('Built at: ' + build.built_at);
  lines.push('Approved items found: ' + build.approved_items_found); // Use the new field
  lines.push('');
  for (var i = 0; i < build.results.length; i++) {
    var r = build.results[i];
    lines.push('## ' + r.item_code + ' — ' + r.item_name);
    lines.push('Status: ' + r.status);
    lines.push('Checked off: ' + (r.checked_off ? 'YES' : 'NO'));
    lines.push('Scaffold: ' + r.scaffold_folder_url);
    lines.push('');
  }
  return lines.join('
');
}

function noBuildReportMarkdown_(report) {
  var lines = [];
  lines.push('# BUILD REPORT: NO APPROVED ITEMS');
  lines.push('');
  lines.push('Generated at: ' + new Date().toISOString());
  lines.push('');
  lines.push('Approved items found: ' + report.approved_items_found);
  lines.push('');
  lines.push('Message: ' + report.message);
  return lines.join('\n');
}

// Placeholder for CT (Constants) object and other helper functions to avoid errors when reviewing.
// In a real Apps Script environment, these would be defined globally or passed via context.
var CT = {
  PROOF_EXTENSIONS: ['png', 'jpg', 'jpeg'],
  FINAL_EXTENSIONS: ['pdf', 'png', 'jpg', 'jpeg'],
  STATUS: {SOURCE_FOUND: 'SOURCE FOUND', MISSING: 'MISSING', DUPLICATE: 'DUPLICATE'},
  APPROVAL_STATUS: {
    WAITING: 'WAITING_APPROVAL',
    APPROVED: 'APPROVED',
    NEEDS_REPAIR: 'NEEDS_REPAIR',
    REJECTED: 'REJECTED',
    MISSING: 'MISSING_FIND_LATER',
    DUPLICATE: 'DUPLICATE_REVIEW',
    BUILT: 'BUILT_VERIFIED',
    GOOD: 'GOOD',
    BUILT_READY: 'BUILT_READY'
  },
  SCAFFOLD_SCAN_FILE: '01_FULL_SCAFFOLD_SCAN.json',
  ARCHIVE_SCAN_FILE: '02_FULL_ARCHIVE_SCAN.json',
  MATCHES_FILE: '03_PRODUCT_MATCHES.json'
};

function getContext_() { return { controlFolder: {} }; }
function readJson_(folder, file) { return { items: [], files: [] }; }
function writeJson_(folder, file, data) { console.log('Writing JSON:', file); }
function createText_(folder, file, content) { console.log('Creating text file:', file); }
function matchesMarkdown_(matches) { return 'Generated matches markdown'; }
function stamp_() { return '20230101_120000'; }
function normalize_(s) { return String(s || '').toLowerCase().replace(/[^a-z0-9]/g, ''); }
function getExtension_(name) { var m = String(name || '').toLowerCase().match(/\.([a-z0-9]+)$/); return m ? m[1] : ''; }
function hasCodeMatch_(raw, code) { return raw.includes(normalize_(code)); }
function isGenericToken_(token) { return false; }

// Dummy function for updateApprovalSheetAfterBuild_ to avoid errors on review
function updateApprovalSheetAfterBuild_(sheet, results) {
  console.log('Dummy updateApprovalSheetAfterBuild_ called.');
}

// Dummy function for getApprovalSheet_ to avoid errors on review
function getApprovalSheet_(controlFolder) {
  console.log('Dummy getApprovalSheet_ called.');
  return {}; // Return a dummy object if needed by readApprovalRows_ or other functions
}

// Dummy function for readApprovalRows_ to avoid errors on review
function readApprovalRows_(sheet) {
  console.log('Dummy readApprovalRows_ called.');
  // Return dummy data for testing purposes if needed
  return [
    { item_code: '12.34', approval_status: 'APPROVED' },
    { item_code: '56.78', approval_status: 'GOOD' },
    { item_code: '90.12', approval_status: 'BUILT_READY' }
  ];
}

// Dummy function for getScaffoldItemFolders_ to avoid errors on review
function getScaffoldItemFolders_(folder) {
  console.log('Dummy getScaffoldItemFolders_ called.');
  return { selected: {}, visualProof: {}, sourceNotes: {}, rejected: {}, lockStatus: {}, precheck: {}, variations: {}, finalDestination: {} };
}

// Dummy function for buildApprovedItem_ to avoid errors on review
function buildApprovedItem_(item, itemFolder, folders) {
  console.log('Dummy buildApprovedItem_ called for item:', item.item_code);
  return { item_code: item.item_code, status: 'ADDED', copied_files: [], missing: [], warnings: [], checked_off: true };
}

SyntaxError: invalid syntax (580688940.py, line 1)